In [68]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation
nw_Cooling_net = Network()
nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
Cooling_net_compressor = Compressor("compresor")
Cooling_net_condenser = Condenser("condensador")
Cooling_net_valve = Valve("valvula_expansion")
Cooling_net_evaporator = HeatExchanger("evaporador")
Cooling_net_cc=CycleCloser('CycleCloser')
Cooling_net_source_consumer=Source("Source_Consumer ")
Cooling_net_sink_consumer=Sink("Sink_consumer")
Cooling_net_source_reseau=Source("Source_Reseau")
Cooling_net_sink_reseau=Sink("Sink_Reseau")
Cooling_net_c0=Connection(Cooling_net_valve, 'out1', Cooling_net_cc, 'in1', label='0')
Cooling_net_c1 = Connection(Cooling_net_cc, 'out1', Cooling_net_evaporator, 'in2', label='1')
Cooling_net_c2 = Connection(Cooling_net_evaporator, 'out2', Cooling_net_compressor, 'in1', label='2')
Cooling_net_c3 = Connection(Cooling_net_compressor, 'out1', Cooling_net_condenser, 'in1', label='3')
Cooling_net_c4 = Connection(Cooling_net_condenser, 'out1', Cooling_net_valve, 'in1', label='4')
Cooling_net_c5=Connection(Cooling_net_condenser, 'out2',Cooling_net_sink_consumer, 'in1', label='5')
Cooling_net_c6=Connection(Cooling_net_source_consumer, 'out1',Cooling_net_condenser , 'in2', label='6')
Cooling_net_c7=Connection(Cooling_net_source_reseau, 'out1',Cooling_net_evaporator , 'in1', label='7')
Cooling_net_c8=Connection(Cooling_net_evaporator, 'out1',Cooling_net_sink_reseau , 'in1', label='8')
nw_Cooling_net.add_conns(Cooling_net_c0,  Cooling_net_c1,  Cooling_net_c2,  Cooling_net_c3,  Cooling_net_c4 , Cooling_net_c5,  Cooling_net_c6, Cooling_net_c7, Cooling_net_c8)
def my_ude(ude):
    return ude.conns[0].calc_T_dew() +5-ude.conns[1].calc_T()
def my_ude_dependents(ude):
    c1, c2 = ude.conns
    return [c1.p,c1.h, c2.p,c2.h]
def my_ude_2(ude):
    return ude.conns[0].calc_T() +5-ude.conns[1].calc_T()
def my_ude_dependents_2(ude):
    c1, c2 = ude.conns
    return [c1.p,c1.h, c2.p,c2.h]
ude = UserDefinedEquation(
'my ude', my_ude, my_ude_dependents, conns=[Cooling_net_c1, Cooling_net_c2])
ude_2 = UserDefinedEquation(
'my_ude_2', my_ude_2, my_ude_dependents_2, conns=[Cooling_net_c8, Cooling_net_c7])
nw_Cooling_net.add_ude(ude)
nw_Cooling_net.add_ude(ude_2)
Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5,Q=-15000)
Cooling_net_compressor.set_attr(eta_s=0.95)
Cooling_net_c2.set_attr(fluid={"R134a": 1})
            # 6. Parámetros del Consumidor 
Cooling_net_c5.set_attr(T=60, p=3, fluid={"water": 1})
Cooling_net_c6.set_attr(T=50)
# 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
Cooling_net_c7.set_attr(T=  6.537e+01 , p=2.5, fluid={"water": 1})
import CoolProp.CoolProp as CP
T_triple = CP.Props1SI("Ttriple", "R134a")        # Triple point temperature (K)
p_triple = CP.Props1SI("ptriple", "R134a")        # Triple point pressure (Pa)
T_critical = CP.Props1SI("T_critical", "R134a")  # Critical temperature (K)
p_critical = CP.Props1SI("p_critical", "R134a")  # Critical pressure (Pa)
h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, "R134a")
T_max_K =  T_critical*0.9
p_high = min(p_critical * 0.9, 30e5) 
h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, "R134a")
nw_Cooling_net._set_p_range([p_triple, p_high])
nw_Cooling_net._set_h_range([h_min,h_max])
nw_Cooling_net.solve('design')


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 5.80e+05   | 2 %        | 2.91e+00   | 2.42e+05   | 2.89e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 4.08e+05   | 4 %        | 2.06e+00   | 9.10e+03   | 2.07e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 2.73e+05   | 6 %        | 5.17e-01   | 1.74e+01   | 8.93e+04   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 6.25e+04   | 13 %       | 2.20e+00   | 7.36e-05   | 2.21e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 8.54e+01   | 45 %       | 3.61e-04   | 0.00e+00   | 1.17e-01   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 5.15e-05   | 100 %      | 4.18e-10   | 4.94e-09   | 3.67e-04   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 7     | 2.05e-07   | 100 %      | 2.19e-10   | 2.47e-09   | 3.67e-04   | 0.00e+00   | 0.00e+00   | 0.0

In [69]:
nw_Cooling_net.print_results()


##### RESULTS (CycleCloser) #####
+-------------+------------------+-------------------+
|             |   mass_deviation |   fluid_deviation |
|-------------+------------------+-------------------|
| CycleCloser |         0.00e+00 |          0.00e+00 |
+-------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+----------+----------+-----------+----------|
| compresor | 5.18e+02 | 1.26e+00 | -3.85e+00 | 9.50e-01 |
+-----------+----------+----------+-----------+----------+
##### RESULTS (Condenser) #####
+-------------+-----------+----------+----------+----------+----------+----------+----------+-----------+----------+----------+----------+----------+------------+----------+------------+----------+-----------+------------+-----------+
|             |         Q |       UA |       kA |   td_log |     lmtd |    ttd_u |    ttd_l |  

In [70]:
nw_Heating_net = Network()
nw_Heating_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
Heating_net_compressor = Compressor("compresor")
Heating_net_condenser = Condenser("condensador")
Heating_net_valve = Valve("valvula_expansion")
Heating_net_evaporator = HeatExchanger("evaporador")
Heating_net_cc=CycleCloser('CycleCloser')
Heating_net_source_consumer=Source("Source_Consumer ")
Heating_net_sink_consumer=Sink("Sink_consumer")
Heating_net_source_reseau=Source("Source_Reseau")
Heating_net_sink_reseau=Sink("Sink_Reseau")
Heating_net_c0=Connection(Heating_net_valve, 'out1', Heating_net_cc, 'in1', label='0')
Heating_net_c1 = Connection(Heating_net_cc, 'out1', Heating_net_evaporator, 'in2', label='1')
Heating_net_c2 = Connection(Heating_net_evaporator, 'out2', Heating_net_compressor, 'in1', label='2')
Heating_net_c3 = Connection(Heating_net_compressor, 'out1',Heating_net_condenser, 'in1', label='3')
Heating_net_c4 = Connection(Heating_net_condenser, 'out1', Heating_net_valve, 'in1', label='4')
Heating_net_c5 = Connection(Heating_net_evaporator, 'out1', Heating_net_sink_consumer, 'in1', label='5')
Heating_net_c6 = Connection(Heating_net_source_consumer, 'out1', Heating_net_evaporator, 'in1', label='6')  
Heating_net_c7 = Connection(Heating_net_source_reseau, 'out1', Heating_net_condenser, 'in2', label='7')
Heating_net_c8 = Connection(Heating_net_condenser, 'out2', Heating_net_sink_reseau, 'in1', label='8')
nw_Heating_net.add_conns(Heating_net_c0,  Heating_net_c1,  Heating_net_c2,  Heating_net_c3,  Heating_net_c4 , Heating_net_c5,  Heating_net_c6, Heating_net_c7, Heating_net_c8)
Heating_ude = UserDefinedEquation(
'my_ude', my_ude, my_ude_dependents, conns=[Heating_net_c1, Heating_net_c2])
Heating_ude_2 = UserDefinedEquation(
                'my_ude_2', my_ude_2, my_ude_dependents_2, conns=[Heating_net_c7, Heating_net_c8])
nw_Heating_net.add_ude(Heating_ude)
nw_Heating_net.add_ude(Heating_ude_2 )
Heating_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5,Q=-1500)
Heating_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
Heating_net_compressor.set_attr(eta_s=0.95)
Heating_net_c2.set_attr(fluid={"R134a": 1})
# 6. Parámetros del Consumidor 
Heating_net_c5.set_attr(T=25, p=3, fluid={"water": 1})
Heating_net_c6.set_attr(T=35)
# 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °
Heating_net_c8.set_attr(T=5.688e+01, p=2.5, fluid={"water": 1})
import CoolProp.CoolProp as CP
T_triple = CP.Props1SI("Ttriple", "R134a")        # Triple point temperature (K)
p_triple = CP.Props1SI("ptriple", "R134a")        # Triple point pressure (Pa)
T_critical = CP.Props1SI("T_critical", "R134a")  # Critical temperature (K)
p_critical = CP.Props1SI("p_critical", "R134a")  # Critical pressure (Pa)
h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, "R134a")
T_max_K =  T_critical*0.9
p_high = min(p_critical * 0.9, 30e5) 
h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, "R134a")
nw_Heating_net._set_p_range([p_triple, p_high])
nw_Heating_net._set_h_range([h_min,h_max])
nw_Heating_net.solve('design')


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 4.95e+06   | 0 %        | 7.63e+01   | 1.63e+06   | 1.60e+06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 3.24e+07   | 0 %        | 4.49e+02   | 3.16e+05   | 2.06e+06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 1.37e+08   | 0 %        | 3.23e+02   | 1.45e+05   | 4.57e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 3.08e+07   | 0 %        | 7.83e+01   | 3.85e+04   | 2.73e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 2.88e+07   | 0 %        | 1.88e+03   | 1.84e+05   | 2.10e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 4.51e+07   | 0 %        | 2.06e+03   | 3.99e+04   | 5.76e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 7     | 7.34e+03   | 23 %       | 5.45e-01   | 1.15e+03   | 3.42e+01   | 0.00e+00   | 0.00e+00   | 0.0